[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Preprocessing


## What you will be able to do

Put numeric columns on one scale and turn text columns into numbers, learning how from the training
rows alone, so that a model can use every column of the stations and the test set stays unseen.

## The idea

### The problem

The stations' five numeric columns are measured in different units. Latitude runs from 58 to 71
degrees, elevation from near sea level to over 1,200 meters, and the installation year from 1950 to
2024. A decision tree is unaffected by the units. A logistic regression, which weighs every column
in one sum and finds its weights step by step, can run out of steps before it settles when one
column's numbers are a thousand times another's. Its coefficients then also come out in each
column's own units, so they cannot be compared with each other.

Two columns are text, `terrain` and `owner`. The **Your First Model** notebook gave `terrain` to a
linear model and got an error, because a model computes with numbers. Replacing coastal, fjord,
inland and mountain with 0, 1, 2 and 3 would make them numbers, and would also tell the model that
inland sits between fjord and mountain, with equal steps between them, which the terrains do not
have.

Both fixes learn something from the data, a column's mean and spread or its list of categories.
Learned from all 1,200 stations, they would carry information about the test stations into training,
and the test score would stop measuring stations the model had not seen.

### What preprocessing is

> A **transformer** is an estimator whose `fit` learns how to change the data and whose
> **`transform`** changes it. **`StandardScaler`** learns each column's mean and standard deviation
> and turns every value into its distance from the mean, counted in standard deviations.
> **`OneHotEncoder`** learns each text column's categories and turns the column into one column of
> 0s and 1s for every category, with a 1 in the column of the row's own category.

### Why it works that way

A transformer is fitted once, on the training rows, and then transforms the training rows and the
test rows with the same learned numbers. A test station is treated exactly as a new station would
be, measured against the training set's means and encoded with the training set's categories. That
is also why the test set's scaled columns do not come out with a mean of exactly 0.

Scaling changes nothing a tree can see. A tree splits one column at a time at a threshold, and
stretching a column moves the threshold with it, so the same rows land on the same side. A logistic
regression combines all the columns, and scaled columns let its solver reach the answer in a few
steps and make its coefficients comparable, one unit being one standard deviation for every column.
`LogisticRegression` also holds its coefficients back from growing large, by an amount set by `C`,
1.0 unless you choose another, where a smaller `C` holds them back more. That penalty treats every
coefficient alike, so it is fair to the columns only when they share a scale.

One-hot encoding gives each category a column of its own, so a model learns a separate effect for
coastal, fjord, inland and mountain without assuming any order among them.

### Where this shows up

Models that penalize their coefficients, find them step by step, or measure distances between rows
need scaled inputs, among them logistic regression, ridge and lasso regression, nearest neighbors
and support vector machines. A plain `LinearRegression` solves for its coefficients directly and
makes the same predictions from scaled and unscaled columns, though only scaled coefficients can be
compared with each other. Any table with text columns needs encoding. The **Pipelines** notebook
puts the transformer and the model in one object so the order of `fit` and `transform` cannot go
wrong, and the **Mixed Column Types** notebook applies different transformers to different columns
of one table.

### What this notebook covers

- the numeric columns' different scales
- `StandardScaler` fitted on the training rows and applied to both sets
- a logistic regression on scaled columns, and a tree and a plain linear regression that are
  unaffected
- `OneHotEncoder` on terrain and owner
- scaled numbers and encoded text in one model
- three errors, two of them about what the transformer learned from

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
from sklearn.preprocessing import StandardScaler

heights = [[100.0], [200.0], [300.0]]
scaler = StandardScaler().fit(heights)
print(scaler.mean_, scaler.scale_.round(2))
print(scaler.transform([[200.0], [400.0]]).round(2))
```

```
[200.] [81.65]
[[0.  ]
 [2.45]]
```

`fit` learned a mean of 200 and a standard deviation of 81.65 from three heights. `transform` turned
200 into 0, since it is the mean, and 400, a value it never saw, into 2.45, its distance from the
learned mean in learned standard deviations.

## Setup

Two libraries for the data, scikit-learn and seven of its tools, and the module that catches
warnings.

- `numpy` and `pandas` build the stations, `numpy` also joins scaled and encoded columns side by
  side, and the cell prints scikit-learn's version
- `warnings` and `ConvergenceWarning` catch the warning a model gives when its solver runs out of
  steps
- `train_test_split` divides the stations into training and test sets
- `LogisticRegression` and `DecisionTreeClassifier` predict whether a station has snow on 1 May, and
  `LinearRegression` predicts the January mean, in one section, to show a model scaling leaves alone
- `StandardScaler` scales the numeric columns, and `OneHotEncoder` encodes the text ones

`station_network` makes the same network of 1,200 weather stations in Norway as **The Estimator
API** notebook. The stations are made up, and a generator started from a fixed seed draws them, so
the table is the same on every run. `FEATURES` names the five numeric columns, and `TEXT` the two
text columns, `terrain` and `owner`.

In [1]:
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
TEXT = ["terrain", "owner"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### Columns on different scales

Every example here predicts snow on 1 May, from a split that keeps snow's share the same in both
parts:

In [2]:
snow = network["snow_on_may_1"]
train_rows, test_rows = train_test_split(network.index, test_size=0.25, random_state=0,
                                         stratify=snow)

X_train, X_test = network.loc[train_rows, FEATURES], network.loc[test_rows, FEATURES]
y_train, y_test = snow[train_rows], snow[test_rows]

X_train.describe().loc[["mean", "std", "min", "max"]].round(2)

,latitude,longitude,elevation_m,coast_km,installed
mean,64.52,17.39,290.69,24.59,1987.76
std,3.72,5.72,186.23,24.28,21.44
min,58.01,5.48,12.00,0.00,1950.00
max,70.96,29.15,1224.00,153.00,2024.00


Splitting the index, the table's row labels, keeps the text columns available for the same rows
later on. The means range from about 17 to almost 2,000, and the standard deviations from under 4 to
186.

### Scaling with what the training rows say

In [3]:
scaler = StandardScaler().fit(X_train)
scaled_train = scaler.transform(X_train)
scaled_test = scaler.transform(X_test)

print("learned means:  ", scaler.mean_.round(2))
print("learned spreads:", scaler.scale_.round(2))
print("training means after scaling are 0:", np.allclose(scaled_train.mean(axis=0), 0))
print("test means after scaling:", scaled_test.mean(axis=0).round(2))

learned means:   [  64.52   17.39  290.69   24.59 1987.76]
learned spreads: [  3.72   5.72 186.13  24.27  21.42]
training means after scaling are 0: True
test means after scaling: [-0.08 -0.08  0.05  0.01 -0.04]


The scaler learned from the 900 training stations only, so the training columns now have a mean of
0, and the test columns are close to 0 without being exactly 0. Its spread for elevation, 186.13, is
a little below the 186.23 that `describe` gave, because pandas divides by one fewer row. The test
stations were measured against the training set's means, as a new station would be. `transform`
returns a NumPy array, and `set_output` asks for a DataFrame with the column names kept:

In [4]:
named = StandardScaler().set_output(transform="pandas").fit(X_train)

named.transform(X_test).head(3).round(2)

,latitude,longitude,elevation_m,coast_km,installed
98,-1.32,-1.42,-1.38,-0.79,-0.97
853,-0.19,-0.45,-0.90,-0.08,0.62
66,-1.37,-1.11,-1.24,-0.51,0.43


### A logistic regression on scaled columns

In [5]:
logistic = LogisticRegression().fit(scaled_train, y_train)

print("steps taken:", logistic.n_iter_[0])
print("test accuracy:", round(logistic.score(scaled_test, y_test), 3))
print(pd.Series(logistic.coef_[0], index=FEATURES).round(2))

steps taken: 12
test accuracy: 0.84
latitude       1.78
longitude      0.65
elevation_m    1.20
coast_km       1.14
installed     -0.09
dtype: float64


The solver settled in 12 steps. Every coefficient is in the same unit now, one standard deviation of
its column, so they can be compared. Latitude pushes hardest toward snow, then elevation and
distance from the coast, and the installation year hardly at all. The Common errors section fits the
same model to the unscaled columns.

### A tree that is unaffected

In [6]:
tree_raw = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)
tree_scaled = DecisionTreeClassifier(max_depth=4, random_state=0).fit(scaled_train, y_train)

raw_accuracy = tree_raw.score(X_test, y_test)
scaled_accuracy = tree_scaled.score(scaled_test, y_test)
same = (tree_raw.predict(X_test) == tree_scaled.predict(scaled_test)).all()

print("accuracy, raw and scaled:", round(raw_accuracy, 3), round(scaled_accuracy, 3))
print("same prediction for every test station:", same)

accuracy, raw and scaled: 0.827 0.827
same prediction for every test station: True


The two trees made the same prediction for every one of the 300 test stations. Scaling moved the
thresholds and kept the order of the values within each column, which is all a tree uses.

### A plain linear regression that is unaffected

A `LinearRegression` predicting the January mean solves for its coefficients in one step, with no
penalty on their size:

In [7]:
jan_train = network.loc[train_rows, "jan_mean"]
raw_linear = LinearRegression().fit(X_train, jan_train)
scaled_linear = LinearRegression().fit(scaled_train, jan_train)

same = np.allclose(raw_linear.predict(X_test), scaled_linear.predict(scaled_test))
print("same predictions from raw and scaled columns:", same)

same predictions from raw and scaled columns: True


Scaling changed its coefficients, which are now per standard deviation, and left every prediction
as it was, so for this model scaling matters only when you want to compare the coefficients.

### Encoding text with OneHotEncoder

In [8]:
encoder = OneHotEncoder(sparse_output=False).fit(network.loc[train_rows, TEXT])

print(encoder.get_feature_names_out())
print(network.loc[test_rows[:3], TEXT].to_numpy().tolist())
print(encoder.transform(network.loc[test_rows[:3], TEXT]))

['terrain_coastal' 'terrain_fjord' 'terrain_inland' 'terrain_mountain'
 'owner_airport' 'owner_met' 'owner_private' 'owner_road']
[['fjord', 'private'], ['inland', 'road'], ['fjord', 'private']]
[[0. 1. 0. 0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 1. 0.]]


The encoder learned four terrains and four owners from the training rows, so two text columns became
eight columns of 0s and 1s. The first test station is in a fjord and owned privately, and its row
has a 1 under `terrain_fjord` and under `owner_private`. `sparse_output=False` returns an ordinary
array. By default the encoder returns a sparse matrix, which stores only the 1s.

### Numbers and text in one model

In [9]:
both_train = np.hstack([scaled_train, encoder.transform(network.loc[train_rows, TEXT])])
both_test = np.hstack([scaled_test, encoder.transform(network.loc[test_rows, TEXT])])

combined = LogisticRegression().fit(both_train, y_train)
print("columns:", both_train.shape[1])
print("test accuracy:", round(combined.score(both_test, y_test), 3))

columns: 13
test accuracy: 0.86


`np.hstack` set the five scaled columns and the eight encoded ones side by side. On this split the
model's test accuracy was 0.86 with terrain and owner and 0.84 without them, six more of the 300
test stations right, which is too small a difference for one split to confirm. Keeping the scaler
and the encoder in step by hand is error-prone, which the **Pipelines** and **Mixed Column Types**
notebooks fix.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/04-preprocessing-solutions.ipynb).

**1.** Fit a `StandardScaler` on the training rows of `elevation_m` alone, and print the scaled values of the first three test stations.

In [10]:
# your code here

**2.** Fit a `LogisticRegression` on the scaled training columns with `C=0.01`, a hundred times the default penalty on large coefficients, and print its test accuracy and coefficients.

In [11]:
# your code here

**3.** Encode `owner` alone with `OneHotEncoder(sparse_output=False)`, fitted on the training rows, and print how many test stations each owner column marks with a 1.

In [12]:
# your code here

**4.** Train a `DecisionTreeClassifier(max_depth=4, random_state=0)` on the combined scaled and encoded columns, and print its test accuracy.

In [13]:
# your code here

## Common errors

### ValueError: Found unknown categories ['research'] in column 1 during transform

In [14]:
new_station = pd.DataFrame({"terrain": ["fjord"], "owner": ["research"]})

encoder.transform(new_station)

ValueError: Found unknown categories ['research'] in column 1 during transform

The encoder learned four owners, and a research station is a fifth it has never seen, so it has no
column for it. Column 1 is `owner`, the second of the two. Tell the encoder what to do with a
category it did not learn:

In [15]:
tolerant = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
tolerant.fit(network.loc[train_rows, TEXT])

print(tolerant.transform(new_station))

[[0. 1. 0. 0. 0. 0. 0. 0.]]


With `handle_unknown="ignore"`, the unknown owner becomes 0 in every owner column, and the terrain
is still encoded. Whether that is right is a decision, since the model then treats the station as
belonging to none of the owners it knows.

### No error, and a test set scaled with its own mean

In [16]:
self_scaled = StandardScaler().fit_transform(X_test)

print("first test station, scaled with the training means:", scaled_test[0].round(2))
print("first test station, scaled with its own set's means:", self_scaled[0].round(2))
changed = (logistic.predict(scaled_test) != logistic.predict(self_scaled)).sum()

print("predictions that changed:", changed)
print("accuracy with the training means:", round(logistic.score(scaled_test, y_test), 3))
print("accuracy with its own means:     ", round(logistic.score(self_scaled, y_test), 3))

first test station, scaled with the training means: [-1.32 -1.42 -1.38 -0.79 -0.97]
first test station, scaled with its own set's means: [-1.23 -1.35 -1.3  -0.82 -0.94]
predictions that changed: 4
accuracy with the training means: 0.84
accuracy with its own means:      0.847


`fit_transform` on the test set learned the test set's own means and spreads, so the same station
came out with other numbers, and four of the 300 predictions changed. The accuracy went up, from
0.84 to 0.847, and nothing in that number says the test set was measured on a different scale from
the one the model learned. A new station arrives alone, with no set of its own to scale it by. Fit
on the training rows, and only `transform` the test rows.

### ConvergenceWarning: lbfgs failed to converge (status=1):

In [17]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    unscaled = LogisticRegression().fit(X_train, y_train)

for warning in caught:
    print(f"{warning.category.__name__}: {str(warning.message).splitlines()[0]}")
print("steps taken:", unscaled.n_iter_[0])
print("test accuracy:", round(unscaled.score(X_test, y_test), 3))

steps taken: 100
test accuracy: 0.837


On the raw columns, the solver used all 100 of its default steps without settling, and the model it
returned is wherever the solver stopped. The rest of the warning suggests a larger `max_iter` or
scaling the data, and scaling is the better fix, since the scaled model settled in 12 steps. A
warning like this one is easy to scroll past, and the model it comes with is unfinished.

## Recap

- Fit a transformer on the training rows, then `transform` the training and test rows with it
- Scale numeric columns with `StandardScaler` for a model with a penalty or a step-by-step solver,
  such as a logistic regression
- Leave the scaling out for a tree, which makes the same splits either way
- Encode text columns with `OneHotEncoder`, and choose `handle_unknown` for categories it never saw
- Read a `ConvergenceWarning` as a model that stopped before it settled

## What is next

The **Pipelines** notebook joins the scaler and the model into one object, and shows the leak a
transformer causes when it learns from rows it should not see.

---

&#8592; **Previous:** [Train and Test](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/03-train-and-test.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Pipelines](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/05-pipelines.ipynb) &#8594;
